In [ ]:
# @title 1. Install dependencies
!pip install -q langgraph langchain langchain-google-genai google-generativeai \
    faiss-cpu pypdf duckduckgo-search fastapi uvicorn pyngrok python-multipart nest-asyncio
print("Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 38.0 MB/s eta 0:00:00
Dependencies installed


In [ ]:
import google.generativeai as genai

print(genai.__version__)

0.8.6


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
# @title 2. Configure Gemini API key
import os
from getpass import getpass

if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass("Paste your Gemini API key: ")

import google.generativeai as genai
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

GEMINI_MODEL = "gemini-2.5-flash"
GEMINI_EMBED_MODEL = "models/gemini-embedding-001"

print("Gemini configured: ", GEMINI_MODEL)

Paste your Gemini API key: ··········
Gemini configured:  gemini-2.5-flash


In [5]:
# @title 3. Gemini calling + embedding helpers
import time

def calling_gemini(prompt: str, system: str = None, temperature: float = 0.3) -> str:
    model = genai.GenerativeModel(
        GEMINI_MODEL,
        system_instruction=system,
    )
    for attempt in range(3):
        try:
            resp = model.generate_content(
                prompt,
                generation_config={"temperature": temperature},
            )
            return resp.text.strip()
        except Exception as e:
            if attempt == 2:
                return f"[Gemini error: {e}]"
            time.sleep(2 * (attempt + 1))

def embed_text(text: str, task_type: str = "retrieval_document"):
    result = genai.embed_content(
        model=GEMINI_EMBED_MODEL,
        content=text,
        task_type=task_type,
    )
    return result["embedding"]

print("Gemini helpers ready")


Gemini helpers ready


In [6]:
result = genai.embed_content(
    model="models/gemini-embedding-001",
    content=["Hello world"],
    task_type="retrieval_document",
)

embedding = result["embedding"][0]

In [7]:
print(embedding)

[-0.026189324, 0.003135996, 0.01789595, -0.08768083, -0.0034239523, 0.018811652, -0.0107915355, -0.010249363, 0.012423585, -0.004823675, -0.014541962, -0.012945177, 0.0064779073, 0.00984695, 0.1701921, -0.00045828905, -0.0029318407, -0.003954109, -0.0018471105, -0.013311674, 0.011583211, 0.0033792453, 0.016849266, -0.0033075185, -0.019205093, 0.005389925, 0.019858744, 0.003008713, 0.013527092, -0.0059906575, -0.000116348885, 0.015598001, -0.0058128904, 0.013240212, -0.00076199905, 0.014480422, 0.016065778, 0.0056224805, -0.00080315315, -0.00016009493, -0.0023708004, 0.00052494026, 0.00199694, -0.01344465, 0.016736273, 0.014849355, 0.004278531, -0.016885038, 0.009061078, 0.009466289, -0.011331954, -0.0032028116, -0.018081794, -0.28672966, -0.022282565, 0.0139952935, -0.0018167347, 0.007174819, 0.0010635537, -0.012333255, 0.00085388846, 0.010757289, -0.01200964, -0.022873899, 0.0061556916, 0.003817444, -0.0023144411, 0.00495198, -0.021154622, 0.0018120882, 0.009485222, 0.00612637, 0.0034

In [ ]:
%pip install -U google-generativeai

In [8]:
import requests
print(requests.get("https://generativelanguage.googleapis.com").status_code)

404


In [9]:
# @title 4. Document ingestion & FAISS vector store
import faiss
import numpy as np
from pypdf import PdfReader
from dataclasses import dataclass, field
from typing import List, Dict


In [10]:
EMBED_DIM = 3072

In [11]:
@dataclass
class DocChunk:
    text: str
    source: str
    chunk_id: int


class FaissStore:
    def __init__(self, dim: int = EMBED_DIM):
        self.index = faiss.IndexFlatIP(dim)
        self.chunks: List[DocChunk] = []

    def _normalize(self, vec):
        v = np.array(vec, dtype="float32")
        norm = np.linalg.norm(v)
        return v / norm if norm > 0 else v

    def add_document(self, text: str, source: str, chunk_size: int = 1000, overlap: int = 150):
      chunks = self._split(text, chunk_size, overlap)
      print(f"Splitting complete: {len(chunks)} chunks")
      vectors = []
      for i, c in enumerate(chunks):
        print(f"  Embedding internal chunk {i+1}/{len(chunks)}...", flush=True)
        emb = embed_text(c, task_type="retrieval_document")
        print(f"  Embedding received ({len(emb)} dimensions)", flush=True)
        vectors.append(self._normalize(emb))
        self.chunks.append(DocChunk(text=c, source=source, chunk_id=i))
        if vectors:
          print("Adding vectors to FAISS...", flush=True)
          self.index.add(np.vstack(vectors))
          print(f"Indexed {len(chunks)} chunks from '{source}'")

    def _split(self, text: str, chunk_size: int, overlap: int) -> List[str]:
        text = " ".join(text.split())
        chunks, start = [], 0
        while start < len(text):
            end = start + chunk_size
            chunks.append(text[start:end])
            start = end - overlap
        return [c for c in chunks if c.strip()]

    def search(self, query: str, k: int = 4) -> List[DocChunk]:
        if self.index.ntotal == 0:
            return []
        q = self._normalize(embed_text(query, task_type="retrieval_query")).reshape(1, -1)
        scores, idxs = self.index.search(q, min(k, self.index.ntotal))
        return [self.chunks[i] for i in idxs[0] if i != -1]


vector_store = FaissStore()
print(vector_store)
print("FAISS store ready (empty)")


FAISS store ready (empty)


In [12]:
!pip install -U google-generativeai

In [13]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 3.0 MB/s eta 0:00:00


In [14]:
import os
import time
from PyPDF2 import PdfReader

def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap

    return chunks


def ingest_pdf(path: str):
    try:
        print(f"Opening PDF: {path}")

        reader = PdfReader(path)
        text = "\n".join(page.extract_text() or "" for page in reader.pages)

        if not text.strip():
            print("Warning: No text extracted from the PDF.")
            return

        chunks = chunk_text(text)

        print(f"Characters: {len(text)}")
        print(f"Chunks: {len(chunks)}")

        for i, chunk in enumerate(chunks):
            print(f"\nEmbedding chunk {i+1}/{len(chunks)}")

            start = time.time()

            try:
                vector_store.add_document(
                    chunk,
                    source=f"{os.path.basename(path)}_chunk_{i}"
                )

                print(f" Chunk {i+1} indexed in {time.time()-start:.2f}s")

            except Exception as e:
                print(f" Failed on chunk {i+1}")
                print(e)
                raise

        print("\nFinished indexing.")

    except Exception as e:
        print("Error during PDF ingestion:")
        print(e)


def ingest_text(text: str, source: str):
    try:
        print(f"Adding document: {source}")

        vector_store.add_document(text, source=source)

        print("Done.")

    except Exception as e:
        print(e)

In [16]:
ingest_pdf("21-ijaema-july-4189.pdf")

Opening PDF: 21-ijaema-july-4189.pdf
Characters: 19418
Chunks: 30

Embedding chunk 1/30
Splitting complete: 1 chunks
  Embedding internal chunk 1/1...
  Embedding received (3072 dimensions)
Adding vectors to FAISS...
Indexed 1 chunks from '21-ijaema-july-4189.pdf_chunk_0'
✓ Chunk 1 indexed in 0.56s

Embedding chunk 2/30
Splitting complete: 1 chunks
  Embedding internal chunk 1/1...
  Embedding received (3072 dimensions)
Adding vectors to FAISS...
Indexed 1 chunks from '21-ijaema-july-4189.pdf_chunk_1'
✓ Chunk 2 indexed in 0.41s

Embedding chunk 3/30
Splitting complete: 1 chunks
  Embedding internal chunk 1/1...
  Embedding received (3072 dimensions)
Adding vectors to FAISS...
Indexed 1 chunks from '21-ijaema-july-4189.pdf_chunk_2'
✓ Chunk 3 indexed in 0.38s

Embedding chunk 4/30
Splitting complete: 1 chunks
  Embedding internal chunk 1/1...
  Embedding received (3072 dimensions)
Adding vectors to FAISS...
Indexed 1 chunks from '21-ijaema-july-4189.pdf_chunk_3'
✓ Chunk 4 indexed in 0.42

In [17]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [18]:
pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 4.7 MB/s eta 0:00:00


In [ ]:
# @title 6. DuckDuckGo- Web Search tool
from ddgs import DDGS

def web_search(query: str, max_results: int = 5):
    results = []
    try:
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=max_results):
                results.append({
                    "title": r.get("title", ""),
                    "url": r.get("href", ""),
                    "snippet": r.get("body", ""),
                })
    except Exception as e:
        return [{"title": "Search error", "url": "", "snippet": str(e)}]

    return results

print(web_search("machine learning"))

[{'title': 'Machine learning', 'url': 'https://en.wikipedia.org/wiki/Machine_learning', 'snippet': 'Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.Statistics and mathematical optimisation methods compose the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) through unsupervised learning.From a theoretical viewpoint, probably approximately correct learning provides a mathematical and statistical framework for describing machine learning. Most traditional machine learning and deep learning algorithms can be described as empirical risk minimisation und

In [19]:
# @title 7. LangGraph shared state definition
from typing import TypedDict, List, Dict, Optional

class AgentState(TypedDict, total=False):
    query: str                 # request
    task_type: str             # summarize | compare | search_web | search_local | cite | qa
    doc_sources: List[str]
    retrieved: List[Dict]      # chunks pulled from FAISS
    web_results: List[Dict]
    summary: str
    comparison: str
    citations: List[str]
    answer: str
    trace: List[str]           # logs

In [20]:
# @title 8. Agent nodes
ROUTER_SYSTEM = """You are a routing agent for a research assistant.
Classify the user's request into exactly one label:
- "summarize"
- "compare"
- "search_web"
- "search_local"
- "cite"
- "qa"
Reply with ONLY the label, nothing else."""

VALID_LABELS = {"summarize", "compare", "search_web", "search_local", "cite", "qa"}


def router_node(state: AgentState) -> AgentState:
    label = calling_gemini(
        state["query"],
        system=ROUTER_SYSTEM,
        temperature=0,
    )
    label = label.lower().strip().strip('"\'.')

    if label not in VALID_LABELS:
        label = "qa"

    return {
        "task_type": label,
        "trace": state.get("trace", []) + [f"router -> {label}"],
    }


def retriever_node(state: AgentState) -> AgentState:
    chunks = vector_store.search(state["query"], k=4)
    retrieved = [{"text": c.text, "source": c.source} for c in chunks]
    trace = state.get("trace", []) + [f"retriever -> {len(retrieved)} chunk(s) from FAISS"]
    return {"retrieved": retrieved, "trace": trace}


def web_search_node(state: AgentState) -> AgentState:
    results = web_search(state["query"])
    trace = state.get("trace", []) + [f"web_search -> {len(results)} result(s)"]
    return {"web_results": results, "trace": trace}


def summarizer_node(state: AgentState) -> AgentState:
    context = "\n\n".join(f"[{r['source']}]\n{r['text']}" for r in state.get("retrieved", []))
    if not context:
        context = "(no local documents were indexed; summarizing the request itself)"

    prompt = f"""Summarize the following research material clearly and concisely,
using headings and bullet points where helpful.

Material:
{context}

User request: {state['query']}"""

    summary = calling_gemini(prompt, system="You are a precise research summarization assistant.")
    trace = state.get("trace", []) + ["summarizer -> done"]
    return {"summary": summary, "answer": summary, "trace": trace}


def comparator_node(state: AgentState) -> AgentState:
    context = "\n\n".join(f"[{r['source']}]\n{r['text']}" for r in state.get("retrieved", []))
    if not context:
        context = "(no local documents were indexed)"

    prompt = f"""Compare and contrast the following research material with respect to the
user's request. Highlight key similarities, differences, and any notable gaps.

Material:
{context}

User request: {state['query']}"""

    comparison = calling_gemini(prompt, system="You are a careful comparative-analysis assistant.")
    trace = state.get("trace", []) + ["comparator -> done"]
    return {"comparison": comparison, "answer": comparison, "trace": trace}


def citation_node(state: AgentState) -> AgentState:
    retrieved = state.get("retrieved", [])
    citations = sorted({r["source"] for r in retrieved}) if retrieved else []
    if citations:
        answer = "Sources referenced:\n" + "\n".join(f"- {c}" for c in citations)
    else:
        answer = "No local documents have been indexed yet, so no citations are available."
    trace = state.get("trace", []) + [f"citation_gen -> {len(citations)} source(s)"]
    return {"citations": citations, "answer": answer, "trace": trace}


def qa_node(state: AgentState) -> AgentState:
    context_parts = []
    for r in state.get("retrieved", []):
        context_parts.append(f"[{r['source']}]\n{r['text']}")
    for w in state.get("web_results", []):
        context_parts.append(f"[web: {w['title']}]({w['url']})\n{w['snippet']}")
    context = "\n\n".join(context_parts) or "(no supporting context was retrieved)"

    prompt = f"""Answer the user's question using the context below when it is relevant.
If the context does not contain the answer, say so honestly, then answer from
general knowledge if you can.

Context:
{context}

Question: {state['query']}"""

    answer = calling_gemini(prompt, system="You are a helpful, accurate research assistant.")
    trace = state.get("trace", []) + ["qa -> done"]
    return {"answer": answer, "trace": trace}

print("Agent nodes ready: router, retriever, web_search, summarizer, comparator, citation_gen, qa")


Agent nodes ready: router, retriever, web_search, summarizer, comparator, citation_gen, qa


In [21]:
print(type(calling_gemini))
print(calling_gemini)

<class 'function'>
<function calling_gemini at 0x79d4db4f6d40>


In [22]:
# @title 9. Build & compile the LangGraph agent graph
from langgraph.graph import StateGraph, END

graph = StateGraph(AgentState)

# Nodes
graph.add_node("router", router_node)
graph.add_node("retriever", retriever_node)
graph.add_node("web_search", web_search_node)
graph.add_node("summarizer", summarizer_node)
graph.add_node("comparator", comparator_node)
graph.add_node("citation_gen", citation_node)
graph.add_node("qa", qa_node)

graph.set_entry_point("router")


def route_initial(state: AgentState) -> str:
    task = state.get("task_type", "qa")
    if task == "search_web":
        return "web_search"
    if task in {"summarize", "compare", "search_local"}:
        return "retriever"
    return "qa"


graph.add_conditional_edges(
    "router",
    route_initial,
    {
        "retriever": "retriever",
        "web_search": "web_search",
        "qa": "retriever",   # "cite"/"qa" tasks still retrieve context first
    },
)


# retrieved context
def route_after_retrieval(state: AgentState) -> str:
    task = state.get("task_type", "qa")
    return {
        "summarize": "summarizer",
        "compare": "comparator",
        "cite": "citation_gen",
        "search_local": "qa",
        "qa": "qa",
        "search_web": "qa",  # fallback
    }.get(task, "qa")


graph.add_conditional_edges(
    "retriever",
    route_after_retrieval,
    {
        "summarizer": "summarizer",
        "comparator": "comparator",
        "citation_gen": "citation_gen",
        "qa": "qa",
    },
)

graph.add_edge("web_search", "qa")

graph.add_edge("summarizer", END)
graph.add_edge("comparator", END)
graph.add_edge("citation_gen", END)
graph.add_edge("qa", END)

app_graph = graph.compile()
print("LangGraph compiled successfully")


def run_assistant(question: str) -> dict:
    """Run the full agent graph on a single question and return the final state."""
    initial_state: AgentState = {"query": question, "trace": []}
    return app_graph.invoke(initial_state)


LangGraph compiled successfully


In [23]:
# @title 10. Launch Gradio UI (upload a PDF, then ask)
import gradio as gr

def upload_and_index(file):
    if file is None:
        return "No file uploaded."
    try:
        ingest_pdf(file)
        return f"Indexed. Total chunks in store: {vector_store.index.ntotal}"
    except Exception as e:
        return f"Error indexing file: {e}"


def chat(question, history):
    if not question or not question.strip():
        return "Please enter a question.", ""

    result = run_assistant(question)
    answer = result.get("answer") or "No answer was generated."

    sources = sorted({r["source"] for r in result.get("retrieved", [])})
    if sources:
        answer += "\n\n**Sources:** " + ", ".join(sources)

    trace = " -> ".join(result.get("trace", []))
    return answer, trace


with gr.Blocks(title="Research Paper Chat Assistant") as demo:
    gr.Markdown("# Research Paper Chat Assistant\nUpload a PDF, then ask a question about it. "
                "The agent will classify your request (summarize / compare / cite / web search / Q&A) "
                "and answer using the indexed document(s).")

    with gr.Row():
        file_input = gr.File(label="Upload PDF", file_types=[".pdf"])
        upload_status = gr.Textbox(label="Index status", interactive=False)
    file_input.change(upload_and_index, inputs=file_input, outputs=upload_status)

    question_box = gr.Textbox(label="Ask a question", placeholder="e.g. What is the main contribution of this paper?")
    ask_btn = gr.Button("Ask", variant="primary")

    answer_box = gr.Textbox(label="Answer", lines=8)
    trace_box = gr.Textbox(label="Agent trace (debug)", lines=2)

    ask_btn.click(chat, inputs=[question_box, gr.State([])], outputs=[answer_box, trace_box])
    question_box.submit(chat, inputs=[question_box, gr.State([])], outputs=[answer_box, trace_box])

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://686919b2a94c102fc4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Opening PDF: /tmp/gradio/d74df27f01d35ff50cf3cc91eb02631ffdafec4fcdea546550dec0d97a7c08ab/21-ijaema-july-4189.pdf
Characters: 19418
Chunks: 30

Embedding chunk 1/30
Splitting complete: 1 chunks
  Embedding internal chunk 1/1...
  Embedding received (3072 dimensions)
Adding vectors to FAISS...
Indexed 1 chunks from '21-ijaema-july-4189.pdf_chunk_0'
✓ Chunk 1 indexed in 0.43s

Embedding chunk 2/30
Splitting complete: 1 chunks
  Embedding internal chunk 1/1...
  Embedding received (3072 dimensions)
Adding vectors to FAISS...
Indexed 1 chunks from '21-ijaema-july-4189.pdf_chunk_1'
✓ Chunk 2 indexed in 0.41s

Embedding chunk 3/30
Splitting complete: 1 chunks
  Embedding internal chunk 1/1...
  Embedding received (3072 dimensions)
Adding vectors to FAISS...
Indexed 1 chunks from '21-ijaema-july-4189.pdf_chunk_2'
✓ Chunk 3 indexed in 0.36s

Embedding chunk 4/30
Splitting complete: 1 chunks
  Embedding internal chunk 1/1...
  Embedding received (3072 dimensions)
Adding vectors to FAISS...
Inde

/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 386, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2280, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1657, in call_function
    prediction = await anyio.to_thread.run_sync(  # type:

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://686919b2a94c102fc4.gradio.live
